# 04 — Risk Modeling: VaR, Expected Shortfall, and Extreme Value Theory

We estimate Value at Risk (VaR) and Expected Shortfall (ES) three ways — parametric (normal), historical (empirical), and Monte Carlo (correlated multi-asset GBM portfolio) — then bring in Extreme Value Theory (GEV block maxima and POT/GPD) to model the tail directly, motivated by the fat-tails finding from Module 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quant_sims.risk import (
    parametric_var, parametric_es,
    historical_var, historical_es,
    simulate_correlated_returns, monte_carlo_portfolio_var_es,
    block_maxima, fit_gev, gev_return_level,
    fit_gpd, pot_var_es, mean_residual_life,
)
from quant_sims.utils.plotting import plot_var_es_distribution, plot_mean_residual_life

%matplotlib inline

## 1. VaR / ES on a simulated single-asset portfolio

Parametric (Gaussian) vs. historical (empirical) VaR/ES should agree closely when the underlying data really is normal -- let's confirm that on simulated GBM-style returns.

In [ ]:
mu, sigma = 0.0003, 0.015  # roughly daily-scale drift/vol
rng = np.random.default_rng(42)
simulated_returns = rng.normal(mu, sigma, size=200_000)

alpha = 0.95
p_var = parametric_var(mu, sigma, alpha)
p_es = parametric_es(mu, sigma, alpha)
h_var = historical_var(simulated_returns, alpha)
h_es = historical_es(simulated_returns, alpha)

print(f"Parametric VaR/ES: {p_var:.5f} / {p_es:.5f}")
print(f"Historical VaR/ES:  {h_var:.5f} / {h_es:.5f}")

plot_var_es_distribution(simulated_returns, h_var, h_es, alpha, title="Simulated normal returns — VaR/ES")
plt.show()

## 2. Monte Carlo VaR/ES for a correlated multi-asset portfolio

Two assets with different vol and a moderate correlation. This is the realistic use case -- most portfolios hold more than one asset, and correlation matters a lot for aggregate risk.

In [ ]:
weights = np.array([0.6, 0.4])
mus = np.array([0.08, 0.05])
sigmas = np.array([0.22, 0.12])
corr = np.array([[1.0, 0.4], [0.4, 1.0]])

result = monte_carlo_portfolio_var_es(weights, mus, sigmas, corr, T=1/252, alpha=0.99, n_paths=200_000, seed=1)
print(f"1-day 99% Portfolio VaR: {result['var']:.5f}")
print(f"1-day 99% Portfolio ES:  {result['es']:.5f}")

plot_var_es_distribution(result["portfolio_returns"], result["var"], result["es"], alpha=0.99, title="Simulated 2-asset portfolio (correlated GBM) — 1-day VaR/ES")
plt.show()

## 3. VaR / ES on real market data

Now the same historical VaR/ES estimation, but on real S&P 500 daily returns (requires internet; skipped gracefully if unavailable, same pattern as Module 2).

In [ ]:
try:
    import yfinance as yf

    spx = yf.download("^GSPC", period="10y", interval="1d", progress=False)
    prices = spx["Close"].dropna().values.flatten()
    real_returns = np.diff(prices) / prices[:-1]
    have_real_data = len(real_returns) > 500
except Exception as e:
    print(f"Could not fetch real market data ({e}). Skipping this section.")
    have_real_data = False

In [ ]:
if have_real_data:
    real_mu, real_sigma = np.mean(real_returns), np.std(real_returns, ddof=1)

    alpha = 0.99
    real_p_var = parametric_var(real_mu, real_sigma, alpha)
    real_h_var = historical_var(real_returns, alpha)
    real_p_es = parametric_es(real_mu, real_sigma, alpha)
    real_h_es = historical_es(real_returns, alpha)

    print(f"Real S&P 500 daily returns, 99% confidence:")
    print(f"  Parametric VaR/ES: {real_p_var:.5f} / {real_p_es:.5f}")
    print(f"  Historical VaR/ES:  {real_h_var:.5f} / {real_h_es:.5f}")
    print("Note how much larger the historical estimates tend to be -- fat tails (Module 2) mean the")
    print("normal assumption underestimates real tail risk.")

    plot_var_es_distribution(real_returns, real_h_var, real_h_es, alpha, title="Real S&P 500 daily returns — 99% VaR/ES")
    plt.show()

## 4. Extreme Value Theory: Block Maxima / GEV

Split the loss series (negative of returns) into blocks (e.g. monthly) and fit a GEV distribution to the block maxima. This lets us answer "what loss level do we expect to see once every N blocks" (a return level question), using only the extremes rather than the whole distribution.

In [ ]:
if have_real_data:
    losses = -real_returns
    monthly_maxima = block_maxima(losses, block_size=21)  # ~21 trading days per month

    gev_fit = fit_gev(monthly_maxima)
    print(f"GEV fit: shape (xi) = {gev_fit.shape:.3f}, loc = {gev_fit.loc:.5f}, scale = {gev_fit.scale:.5f}")
    print("(xi > 0 indicates a heavy, Frechet-domain tail -- consistent with the fat tails seen in Module 2)")

    for years in [1, 5, 10, 25]:
        return_period_blocks = years * 12  # 12 monthly blocks per year
        level = gev_return_level(gev_fit, return_period_blocks)
        print(f"  {years:>2}-year return level (worst monthly loss expected once every {years} years): {level:.4f}")

## 5. Peaks-Over-Threshold (POT) / GPD

POT uses every exceedance over a threshold, not just one maximum per block -- generally preferred in practice since it uses more of the data. First, a mean residual life plot to help pick a stable threshold.

In [ ]:
if have_real_data:
    candidate_thresholds = np.quantile(losses, np.linspace(0.80, 0.98, 30))
    mrl = mean_residual_life(losses, candidate_thresholds)

    plot_mean_residual_life(candidate_thresholds, mrl, title="Mean Residual Life — look for the roughly linear region")
    plt.show()

In [ ]:
if have_real_data:
    threshold = np.quantile(losses, 0.95)  # top 5% of daily losses as exceedances
    gpd_fit = fit_gpd(losses, threshold)
    print(f"GPD fit: shape (xi) = {gpd_fit.shape:.3f}, scale (beta) = {gpd_fit.scale:.5f}")
    print(f"Threshold = {threshold:.5f}, exceedances = {gpd_fit.n_exceedances} / {gpd_fit.n_total}")

    alpha = 0.99
    evt_var, evt_es = pot_var_es(gpd_fit, alpha)
    normal_var = parametric_var(real_mu, real_sigma, alpha)
    normal_es = parametric_es(real_mu, real_sigma, alpha)
    hist_var = historical_var(real_returns, alpha)
    hist_es = historical_es(real_returns, alpha)

    print(f"\n99% VaR / ES comparison:")
    print(f"  Normal (parametric): {normal_var:.5f} / {normal_es:.5f}")
    print(f"  Historical (empirical): {hist_var:.5f} / {hist_es:.5f}")
    print(f"  EVT (POT/GPD):        {evt_var:.5f} / {evt_es:.5f}")

## Takeaways

- Parametric (normal) VaR/ES agrees closely with historical VaR/ES *only* when the underlying distribution really is close to normal -- true for our simulated GBM returns, but not for real market data.
- On real market data, the normal assumption tends to *underestimate* tail risk (VaR/ES), because it doesn't account for fat tails -- exactly the gap identified in Module 2's excess kurtosis check.
- EVT (both GEV block maxima and POT/GPD) fits the tail directly rather than assuming a global distributional shape, which is precisely why it's used in practice for tail risk: it doesn't need the *whole* distribution to be right, just the extremes.
- POT/GPD is generally preferred over block maxima in practice because it uses every exceedance rather than throwing away all but the single worst observation per block.
- Monte Carlo VaR/ES for a multi-asset portfolio depends heavily on the correlation structure between assets -- this is the natural extension point toward more realistic portfolio risk management.